<a href="https://colab.research.google.com/github/rkahrya1311/amazon-ml-challenge-2026/blob/kishanth%2Fblocking/kishanth_blocking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip -q install duckdb

In [17]:
from pathlib import Path
import duckdb
import pandas as pd

uploaded_folder = Path("/content")
data_folder = Path("/content/cleaned_training_data")
data_folder.mkdir(exist_ok=True)

file_names = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
]

# Check that the uploaded files are present, then copy them while replacing
# only invalid UTF-8 bytes. This keeps the rows instead of skipping them.
for file_name in file_names:
    uploaded_file = uploaded_folder / file_name

    if not uploaded_file.exists():
        raise FileNotFoundError(f"File not found: {uploaded_file}")

    cleaned_file = data_folder / file_name

    with uploaded_file.open("rb") as input_file:
        with cleaned_file.open(
            "w", encoding="utf-8", newline=""
        ) as output_file:
            for line in input_file:
                output_file.write(line.decode("utf-8", errors="replace"))

temporary_folder = Path("/content/duckdb_temporary")
temporary_folder.mkdir(exist_ok=True)

connection = duckdb.connect()
connection.execute("SET threads = 4")
connection.execute("SET memory_limit = '8GB'")
connection.execute(
    f"SET temp_directory = '{temporary_folder.as_posix()}'"
)

def read_tab_separated_file(file_path):
    escaped_path = file_path.as_posix().replace("'", "''")
    return (
        f"read_csv('{escaped_path}', "
        "delim='\\t', "
        "header=true, "
        "all_varchar=true, "
        "quote='', "
        "strict_mode=false, "
        "null_padding=true)"
    )

# Read the three training sources.
for source_number in (1, 2, 3):
    file_path = data_folder / f"train_source{source_number}.tsv"
    connection.execute(f"""
        CREATE OR REPLACE VIEW training_source{source_number} AS
        SELECT * FROM {read_tab_separated_file(file_path)}
    """)

# Count records in each source and by country.
country_count_queries = []

for source_number in (1, 2, 3):
    country_count_queries.append(f"""
        SELECT
            'S{source_number}' AS source,
            COALESCE(country, 'Missing country') AS country,
            COUNT(*) AS business_records
        FROM training_source{source_number}
        GROUP BY country
    """)

country_counts = connection.execute(
    " UNION ALL ".join(country_count_queries)
    + " ORDER BY country, source"
).df()

source_counts = (
    country_counts
    .groupby("source", as_index=False)["business_records"]
    .sum()
)

print("Number of records in each source:")
display(source_counts)

print("Business records per country and source:")
display(country_counts)

# S2 and S3 are the records compared with each S1 record.
connection.execute("""
    CREATE OR REPLACE TEMP TABLE training_targets AS
    SELECT entity_id, business_name, business_address, country
    FROM training_source2

    UNION ALL

    SELECT entity_id, business_name, business_address, country
    FROM training_source3
""")

# Expand the ground truth so each known match is one row.
ground_truth_file = data_folder / "train_ground_truth.tsv"

connection.execute(f"""
    CREATE OR REPLACE TEMP TABLE ground_truth AS
    SELECT DISTINCT
        ground_truth_rows.source1_entity_id AS source1_entity_id,
        TRIM(matched_values.matched_id) AS target_entity_id
    FROM {read_tab_separated_file(ground_truth_file)}
        AS ground_truth_rows,
    UNNEST(
        string_split(ground_truth_rows.matched_entity_ids, ',')
    ) AS matched_values(matched_id)
    WHERE TRIM(matched_values.matched_id) <> ''
""")

number_of_source1_records = connection.execute(
    "SELECT COUNT(*) FROM training_source1"
).fetchone()[0]

number_of_true_match_pairs = connection.execute(
    "SELECT COUNT(*) FROM ground_truth"
).fetchone()[0]

print(f"Training S1 records: {number_of_source1_records:,}")
print(f"Labeled true-match pairs: {number_of_true_match_pairs:,}")

# Skip any blocking key that points to more than 100 S2/S3 records.
maximum_records_per_blocking_key = 100

name_stop_words = [
    "inc", "incorporated", "corp", "corporation", "llc", "ltd",
    "limited", "co", "company", "plc", "gmbh", "the", "and",
]

address_stop_words = [
    "street", "st", "road", "rd", "avenue", "ave", "drive", "dr",
    "suite", "ste", "unit", "floor", "fl", "building", "bldg",
    "block", "near", "opp", "opposite", "behind", "plot", "no",
    "number", "dist", "district",
]

def make_sql_string_list(values):
    return ", ".join(
        "'" + value.replace("'", "''") + "'"
        for value in values
    )

def normalized_text_sql(column_name):
    return (
        f"regexp_replace("
        f"lower(coalesce({column_name}, '')), "
        f"'[^\\p{{L}}\\p{{N}}]+', ' ', 'g'"
        f")"
    )

def create_blocking_keys(blocking_rule):
    for table_name in (
        "target_blocking_keys",
        "source1_blocking_keys",
        "allowed_blocking_keys",
        "candidate_pairs",
    ):
        connection.execute(f"DROP TABLE IF EXISTS {table_name}")

    if blocking_rule == "B1":
        stop_words = make_sql_string_list(name_stop_words)
        target_name = normalized_text_sql("business_name")
        source1_name = normalized_text_sql("business_name")

        target_keys_query = f"""
            SELECT DISTINCT
                entity_id,
                lower(country) || '|' || name_token AS blocking_key
            FROM training_targets,
            UNNEST(string_split({target_name}, ' '))
                AS name_tokens(name_token)
            WHERE length(name_token) >= 3
              AND name_token NOT IN ({stop_words})
        """

        source1_keys_query = f"""
            SELECT DISTINCT
                entity_id AS source1_entity_id,
                lower(country) || '|' || name_token AS blocking_key
            FROM training_source1,
            UNNEST(string_split({source1_name}, ' '))
                AS name_tokens(name_token)
            WHERE length(name_token) >= 3
              AND name_token NOT IN ({stop_words})
        """

    elif blocking_rule == "B2":
        # First four normalized name characters; this rule does not use country.
        target_name = normalized_text_sql("business_name")
        source1_name = normalized_text_sql("business_name")

        target_keys_query = f"""
            SELECT DISTINCT
                entity_id,
                substr(replace({target_name}, ' ', ''), 1, 4)
                    AS blocking_key
            FROM training_targets
            WHERE length(replace({target_name}, ' ', '')) >= 4
        """

        source1_keys_query = f"""
            SELECT DISTINCT
                entity_id AS source1_entity_id,
                substr(replace({source1_name}, ' ', ''), 1, 4)
                    AS blocking_key
            FROM training_source1
            WHERE length(replace({source1_name}, ' ', '')) >= 4
        """

    elif blocking_rule == "B3":
        stop_words = make_sql_string_list(address_stop_words)
        target_address = normalized_text_sql("business_address")
        source1_address = normalized_text_sql("business_address")

        target_keys_query = f"""
            SELECT DISTINCT
                entity_id,
                lower(country) || '|' || address_token AS blocking_key
            FROM training_targets,
            UNNEST(string_split({target_address}, ' '))
                AS address_tokens(address_token)
            WHERE length(address_token) >= 3
              AND address_token NOT IN ({stop_words})
        """

        source1_keys_query = f"""
            SELECT DISTINCT
                entity_id AS source1_entity_id,
                lower(country) || '|' || address_token AS blocking_key
            FROM training_source1,
            UNNEST(string_split({source1_address}, ' '))
                AS address_tokens(address_token)
            WHERE length(address_token) >= 3
              AND address_token NOT IN ({stop_words})
        """

    else:
        raise ValueError(f"Unknown blocking rule: {blocking_rule}")

    connection.execute(
        "CREATE TEMP TABLE target_blocking_keys AS "
        + target_keys_query
    )
    connection.execute(
        "CREATE TEMP TABLE source1_blocking_keys AS "
        + source1_keys_query
    )

    connection.execute(f"""
        CREATE TEMP TABLE allowed_blocking_keys AS
        SELECT blocking_key
        FROM target_blocking_keys
        GROUP BY blocking_key
        HAVING COUNT(DISTINCT entity_id)
            <= {maximum_records_per_blocking_key}
    """)

    # Count each S1-to-S2/S3 pair once, even if several keys match.
    connection.execute("""
        CREATE TEMP TABLE candidate_pairs AS
        SELECT DISTINCT
            source1_keys.source1_entity_id,
            target_keys.entity_id AS target_entity_id
        FROM source1_blocking_keys AS source1_keys
        JOIN allowed_blocking_keys
            USING (blocking_key)
        JOIN target_blocking_keys AS target_keys
            USING (blocking_key)
    """)

def measure_blocking_rule(blocking_rule):
    create_blocking_keys(blocking_rule)

    candidate_count = connection.execute(
        "SELECT COUNT(*) FROM candidate_pairs"
    ).fetchone()[0]

    retained_true_match_count = connection.execute("""
        SELECT COUNT(*)
        FROM ground_truth
        JOIN candidate_pairs
          ON ground_truth.source1_entity_id =
             candidate_pairs.source1_entity_id
         AND ground_truth.target_entity_id =
             candidate_pairs.target_entity_id
    """).fetchone()[0]

    return {
        "blocking_rule": blocking_rule,
        "candidate_count": candidate_count,
        "average_candidates_per_S1": (
            candidate_count / number_of_source1_records
        ),
        "retained_true_match_count": retained_true_match_count,
        "total_true_match_count": number_of_true_match_pairs,
        "true_match_recall": (
            retained_true_match_count / number_of_true_match_pairs
            if number_of_true_match_pairs
            else 0
        ),
    }

blocking_results = pd.DataFrame([
    measure_blocking_rule("B1"),
    measure_blocking_rule("B2"),
    measure_blocking_rule("B3"),
])

print("Candidate count, average candidates per S1, and true-match recall:")
display(blocking_results)
individual_results = []

for blocking_rule in ("B1", "B2", "B3"):
    rule_result = measure_blocking_rule(blocking_rule)

    saved_table = f"candidate_pairs_{blocking_rule.lower()}"
    connection.execute(f"DROP TABLE IF EXISTS {saved_table}")

    connection.execute(f"""
        CREATE TEMP TABLE {saved_table} AS
        SELECT source1_entity_id, target_entity_id
        FROM candidate_pairs
    """)

    individual_results.append(rule_result)

connection.execute("DROP TABLE IF EXISTS combined_candidate_pairs")

connection.execute("""
    CREATE TEMP TABLE combined_candidate_pairs AS

    SELECT source1_entity_id, target_entity_id
    FROM candidate_pairs_b1

    UNION

    SELECT source1_entity_id, target_entity_id
    FROM candidate_pairs_b2

    UNION

    SELECT source1_entity_id, target_entity_id
    FROM candidate_pairs_b3
""")

combined_candidate_count = connection.execute(
    "SELECT COUNT(*) FROM combined_candidate_pairs"
).fetchone()[0]

combined_retained_true_match_count = connection.execute("""
    SELECT COUNT(*)
    FROM ground_truth
    JOIN combined_candidate_pairs
      ON ground_truth.source1_entity_id =
         combined_candidate_pairs.source1_entity_id
     AND ground_truth.target_entity_id =
         combined_candidate_pairs.target_entity_id
""").fetchone()[0]

combined_result = {
    "blocking_rule": "B1 OR B2 OR B3",
    "candidate_count": combined_candidate_count,
    "average_candidates_per_S1": (
        combined_candidate_count / number_of_source1_records
    ),
    "retained_true_match_count": combined_retained_true_match_count,
    "total_true_match_count": number_of_true_match_pairs,
    "true_match_recall": (
        combined_retained_true_match_count / number_of_true_match_pairs
        if number_of_true_match_pairs
        else 0
    ),
}

comparison_results = pd.DataFrame(
    individual_results + [combined_result]
)

display(comparison_results)

Number of records in each source:


,source,business_records
0,S1,330417
1,S2,312884
2,S3,340855


Business records per country and source:


,source,country,business_records
0,S1,India,132581
1,S2,India,125708
2,S3,India,136914
3,S1,Missing country,1
4,S2,Missing country,1
5,S1,US,197835
6,S2,US,187175
7,S3,US,203941


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,blocking_rule,candidate_count,average_candidates_per_S1,retained_true_match_count,total_true_match_count,true_match_recall
0,B1,5950163,18.008041,8302,1954624,0.004247
1,B2,3567285,10.796312,5586,1954624,0.002858
2,B3,16524139,50.009954,14131,1954624,0.007230
3,B1 OR B2 OR B3,25161438,76.150555,16377,1954624,0.008379
